In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

In [2]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./TinyLlama"

In [3]:
dataset = load_dataset("Abirate/english_quotes")    

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags'],
        num_rows: 2508
    })
})

In [5]:
def format_example(example):

    quote = example["quote"].strip()

    tags = ", ".join(example["tags"]) if isinstance(example["tags"], list) else str(example["tags"])

    text = f"""<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
{quote}<end_of_turn>
<start_of_turn>model
{tags}<end_of_turn>"""

    return {"text": text}


In [6]:
small_dataset = dataset["train"].select(range(100))

In [7]:
small_dataset

Dataset({
    features: ['quote', 'author', 'tags'],
    num_rows: 100
})

In [8]:
train_dataset = small_dataset.map(format_example)

In [9]:
train_dataset

Dataset({
    features: ['quote', 'author', 'tags', 'text'],
    num_rows: 100
})

In [10]:
print(train_dataset["text"][0])

<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
“Be yourself; everyone else is already taken.”<end_of_turn>
<start_of_turn>model
be-yourself, gilbert-perreira, honesty, inspirational, misattributed-oscar-wilde, quote-investigator<end_of_turn>


In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [12]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [13]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [14]:
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, 
    bias="none",
    task_type="CAUSAL_LM"
)

In [15]:
model = get_peft_model(base_model, lora_config)

In [16]:
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [17]:
import torch
print(torch.__version__)
print(torch.version.cuda)          # None => CPU-only build
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no device")

2.11.0+cu130
13.0
True
NVIDIA GB10


In [20]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [21]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)

Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [22]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,2.110857
20,1.163509


TrainOutput(global_step=25, training_loss=1.4790685653686524, metrics={'train_runtime': 10.4127, 'train_samples_per_second': 9.604, 'train_steps_per_second': 2.401, 'total_flos': 61705200033792.0, 'train_loss': 1.4790685653686524, 'entropy': 0.933602511882782, 'num_tokens': 9898.0, 'mean_token_accuracy': 0.8465540051460266, 'epoch': 1.0})

In [23]:
model.save_pretrained(f"{OUTPUT_DIR}/adapter")

In [24]:
tokenizer.save_pretrained(f"{OUTPUT_DIR}/adapter")

('./TinyLlama/adapter/tokenizer_config.json',
 './TinyLlama/adapter/chat_template.jinja',
 './TinyLlama/adapter/tokenizer.json')

In [25]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=2048, out_features=256, bias=False)
            (lora_dropout): ModuleDict(
        

In [27]:
inference_model = PeftModel.from_pretrained(base_model, f"{OUTPUT_DIR}/adapter")

/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [28]:
inference_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [29]:
prompt = """<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
If you want to achieve greatness, stop asking for permission.<end_of_turn>
<start_of_turn>model
"""

In [30]:
inputs = tokenizer(prompt, return_tensors="pt").to(inference_model.device)

In [32]:
inference_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
)
inference_model = PeftModel.from_pretrained(inference_base, f"{OUTPUT_DIR}/adapter")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [33]:
with torch.no_grad():
    output = inference_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [34]:
print(tokenizer.decode(output[0], skip_special_tokens=True))

<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
If you want to achieve greatness, stop asking for permission.<end_of_turn>
<start_of_turn>model
greatness, permission,<end_of_turn>


In [35]:
merged_model = inference_model.merge_and_unload()

/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [36]:
tokenizer.save_pretrained("./merged_full_model")

('./merged_full_model/tokenizer_config.json',
 './merged_full_model/chat_template.jinja',
 './merged_full_model/tokenizer.json')